# 🎲 OmniTale — Autonomous LLM Benchmark & Evaluation
### Long-Context RPG Simulation, RAM/VRAM Tracking & Small Model Profiling for 8GB VRAM GPUs

This Kaggle notebook evaluates small open-weights LLMs (7B – 14B) running under `llama.cpp` (`llama-cpp-python`) on **OmniTale's multi-agent tabletop RPG architecture**.

### 🎯 Benchmark Goals:
1. **Full Game Engine Execution**: Port of OmniTale's prompt pipeline (*Turn 0*, *Dynamic Fate Oracle D100*, *The Judge*, *Lead Narrator*, *Dynamic Lorebook Sync*, *Master's Secret Journal Consolidation*).
2. **Custom Language Support**: Select story language (e.g. *English*, *Italian*, *French*, *German*, *Spanish*).
3. **Autonomous Multi-Turn Play**: An *Auto-Player* agent simulates human roleplay actions with psychological mindset, short/long-term goals, and tactical archetype for **50+ turns**.
4. **Token, Latency & RAM/VRAM Profiling**: Real-time measurement of Prompt/Completion tokens, context accumulation (up to **25K-30K tokens**), generation speed (tokens/sec), stage latency, process RAM, and GPU VRAM usage.
5. **8GB VRAM Sizing Analysis**: Recommendations for running OmniTale locally on domestic hardware (laptops with RTX 3060/3070/4060 8GB VRAM) using KV cache quantization and Flash Attention.

## 1. System & GPU Check

In [ ]:
!nvidia-smi

## 2. Install Dependencies (CUDA-accelerated llama-cpp-python)

In [ ]:
# Install prebuilt CUDA wheel or compile with CUDA support
!pip install -q huggingface_hub psutil pandas matplotlib tabulate tqdm
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

## 3. Select & Download Model (GGUF)
Choose one of the tested single-file GGUF models below or provide your own Hugging Face GGUF repo.

In [ ]:
from huggingface_hub import hf_hub_download

# Tested single-file GGUF presets
MODELS = {
    "qwen2.5-7b": {
        "repo": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "file": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
        "desc": "Qwen 2.5 7B Instruct (Q4_K_M) - Top recommendation for Italian & instruction following"
    },
    "llama3.1-8b": {
        "repo": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "file": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
        "desc": "Llama 3.1 8B Instruct (Q4_K_M) - Native 128k context support"
    },
    "gemma2-9b": {
        "repo": "bartowski/gemma-2-9b-it-GGUF",
        "file": "gemma-2-9b-it-Q4_K_M.gguf",
        "desc": "Gemma 2 9B IT (Q4_K_M) - High literary quality narrative generation"
    }
}

# Select model key (default: qwen2.5-7b)
CHOSEN_KEY = "qwen2.5-7b"
config = MODELS[CHOSEN_KEY]

print(f"Downloading {config['desc']}...")
MODEL_PATH = hf_hub_download(repo_id=config["repo"], filename=config["file"])
print(f"Model downloaded successfully to: {MODEL_PATH}")

## 4. Initialize OmniTale Engine & Session Configuration
Configure context window, language, scenario (including **The Collective Flame**), and KV Cache quantization (saving ~50-70% VRAM!).

In [ ]:
import sys
import os

# Ensure omnitale_engine is in Python path
if "." not in sys.path:
    sys.path.insert(0, ".")

from omnitale_engine import (
    LlamaCppBackend,
    OmniTaleSimulator,
    load_template,
    TEMPLATES
)

# Benchmark configuration
CONTEXT_SIZE = 28000
GPU_LAYERS = -1
OUTPUT_DIR = "./eval_results"
SESSION_LANGUAGE = "English"  # Options: 'English', 'Italian', 'Spanish', 'German', 'French'
TEMPLATE_NAME = "collective-flame"  # Options: 'collective-flame', 'eldoria', 'sector7', 'deepice'

backend = LlamaCppBackend(
    model_path=MODEL_PATH,
    n_ctx=CONTEXT_SIZE,
    n_gpu_layers=GPU_LAYERS,
    flash_attn=True,
    cache_type_k="q8_0",
    cache_type_v="q8_0",
    verbose=False
)

# Load Starter Scenario with language override
story = load_template(TEMPLATE_NAME, language=SESSION_LANGUAGE)
print(f"Loaded Scenario: {story.title} | Genre: {story.genre} | Language: {story.language}")

## 5. Run Multi-Turn Simulation (50 Turns)
The simulation will autonomously execute:
1. **Turn 0**: Stochastic parameter matrix roll & atmospheric opening scene.
2. **Turns 1 to 50**:
   - **Auto-Player**: Generates in-character actions/dialogues driven by character mindset & goals.
   - **Fate Oracle**: Rolls D100 dice for physical/dramatic resolution.
   - **The Judge**: Mechanically adjudicates actions, applies anti-retcon & pacing directives.
   - **Lead Narrator**: Crafts literary world response without echoing player prose.
   - **Background Service (Every 5 turns)**: Asynchronously consolidates Dynamic Lorebook & Master's Secret Journal.
   - **Memory Tracker**: Continuously logs CPU RAM and GPU VRAM consumption.

In [ ]:
TOTAL_TURNS = 50

simulator = OmniTaleSimulator(
    backend=backend,
    story_template=story,
    output_dir=OUTPUT_DIR,
    language=SESSION_LANGUAGE
)

summary = simulator.run_simulation(total_turns=TOTAL_TURNS)

## 6. Story Transcript Viewer
Read the generated RPG adventure directly in the notebook or download the Markdown transcript.

In [ ]:
from IPython.display import display, Markdown

transcript_path = os.path.join(OUTPUT_DIR, "story_transcript.md")
with open(transcript_path, "r", encoding="utf-8") as f:
    md_content = f.read()

# Display first 3000 chars of story in notebook
display(Markdown(md_content[:3000] + "\n\n*(Transcript truncated for preview. Full story saved to `story_transcript.md`)*"))

## 7. Performance, Token & RAM/VRAM Analytics
Visualize token consumption, latency per stage, and memory footprint.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = os.path.join(OUTPUT_DIR, "turns_metrics.csv")
df = pd.read_csv(csv_path)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Tokens per Stage
for stage in df['stage'].unique():
    sdf = df[df['stage'] == stage]
    axes[0, 0].plot(sdf['turn_index'], sdf['total_tokens'], marker='o', label=stage)
axes[0, 0].set_title("Tokens per Turn by Pipeline Stage")
axes[0, 0].set_xlabel("Turn Index")
axes[0, 0].set_ylabel("Total Tokens")
axes[0, 0].grid(True, linestyle='--', alpha=0.6)
axes[0, 0].legend()

# 2. Latency per Stage
for stage in df['stage'].unique():
    sdf = df[df['stage'] == stage]
    axes[0, 1].plot(sdf['turn_index'], sdf['latency_sec'], marker='s', label=stage)
axes[0, 1].set_title("Latency (seconds) per Stage")
axes[0, 1].set_xlabel("Turn Index")
axes[0, 1].set_ylabel("Seconds")
axes[0, 1].grid(True, linestyle='--', alpha=0.6)
axes[0, 1].legend()

# 3. Cumulative Context Growth
turn_max = df.groupby('turn_index')['context_tokens_cumulative'].max()
axes[1, 0].plot(turn_max.index, turn_max.values, color='purple', linewidth=2.5)
axes[1, 0].set_title("Cumulative Story Tokens (Context Window Demand)")
axes[1, 0].set_xlabel("Turn Index")
axes[1, 0].set_ylabel("Cumulative Tokens")
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

# 4. RAM and GPU VRAM Consumption
turn_ram = df.groupby('turn_index')['ram_used_mb'].max()
axes[1, 1].plot(turn_ram.index, turn_ram.values, color='teal', linewidth=2, label="Process RAM (MB)", marker='x')
if 'vram_used_mb' in df.columns and df['vram_used_mb'].notna().any():
    turn_vram = df.groupby('turn_index')['vram_used_mb'].max()
    axes[1, 1].plot(turn_vram.index, turn_vram.values, color='crimson', linewidth=2, label="GPU VRAM (MB)", marker='^')
axes[1, 1].set_title("Memory Consumption (RAM & GPU VRAM in MB)")
axes[1, 1].set_xlabel("Turn Index")
axes[1, 1].set_ylabel("MB")
axes[1, 1].grid(True, linestyle='--', alpha=0.6)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
from tabulate import tabulate

summary_data = [
    ["Scenario & Language", f"{summary['story_title']} ({summary['language']})"],
    ["Total Turns Completed", summary["total_turns_completed"]],
    ["Total Session Duration", f"{summary['total_duration_seconds']} s ({round(summary['total_duration_seconds']/60, 2)} min)"],
    ["Total Prompt Tokens", summary["total_prompt_tokens"]],
    ["Total Completion Tokens", summary["total_completion_tokens"]],
    ["Grand Total Tokens Processed", summary["grand_total_tokens_processed"]],
    ["Overall Speed", f"{summary['overall_tokens_per_second']} tokens/sec"],
    ["Peak Process RAM", f"{summary['peak_process_ram_mb']} MB"],
    ["Peak GPU VRAM", f"{summary.get('peak_gpu_vram_mb', 'N/A')} MB"],
    ["Final Lorebook Size", f"{summary['final_lorebook_characters']} chars"],
    ["Final Master Journal Size", f"{summary['final_journal_characters']} chars"],
]

print(tabulate(summary_data, headers=["Metric", "Value"], tablefmt="fancy_grid"))

## 9. 💻 8GB VRAM Laptop Sizing & Configuration Guide

How to run this setup smoothly on a domestic laptop with **8 GB VRAM** (e.g., RTX 3060/3070/4060 Laptop GPU):

### 📊 VRAM Budget Calculation:
1. **Model Weights (7B Q4_K_M)**: ~4.3 GB VRAM
2. **KV Cache (28K Context at FP16)**: ~7.0 GB ⚠️ *(Exceeds 8GB total with model weights!)*
3. **KV Cache with `q8_0` Quantization**: ~3.5 GB ✅ *(Total: 4.3 + 3.5 = 7.8 GB - Fits in 8GB!)*
4. **KV Cache with `q4_0` Quantization**: ~1.8 GB 🚀 *(Total: 4.3 + 1.8 = 6.1 GB - Comfortable headroom for OS & UI!)*

### 🛠️ Recommended Local Command / Settings (`llama-server` or `Ollama`):
```bash
# Start llama-server with Flash Attention and Q4/Q8 KV cache:
./llama-server \
  -m Qwen2.5-7B-Instruct-Q4_K_M.gguf \
  -c 28000 \
  -ngl 99 \
  --flash-attn \
  --cache-type-k q8_0 \
  --cache-type-v q8_0 \
  --port 8080
```

Then in OmniTale's UI settings, point the LLM URL to: `http://localhost:8080/v1`.